# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [12]:
import pandas as pd 
df=pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv")
df["Response"].unique()



array(['No', 'Yes', nan], dtype=object)

In [14]:
low_claim = df["Total Claim Amount"]< 1000
response=df["Response"] == "Yes"

In [46]:
filter_df=df[low_claim&response]
filter_df.columns

Index(['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value',
       'Response', 'Coverage', 'Education', 'Effective To Date',
       'EmploymentStatus', 'Gender', 'Income', 'Location Code',
       'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim',
       'Months Since Policy Inception', 'Number of Open Complaints',
       'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type',
       'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size',
       'Vehicle Type'],
      dtype='object')

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [24]:
grouped=df.groupby(["Policy Type","Gender"]).agg({"Monthly Premium Auto":"mean","Customer Lifetime Value":"mean","Total Claim Amount":"mean"})
grouped

Monthly Premium Auto  Customer Lifetime Value  \
Policy Type    Gender                                                  
Corporate Auto F                  91.384615              7980.306825   
               M                  94.764249              7750.741082   
Personal Auto  F                  93.153179              8074.660516   
               M                  93.301056              7971.386285   
Special Auto   F                  93.563025              8460.398042   
               M                  93.197044              9010.601583   

                       Total Claim Amount  
Policy Type    Gender                      
Corporate Auto F               397.799287  
               M               462.223565  
Personal Auto  F               413.239658  
               M               459.919476  
Special Auto   F               458.139623  
               M               420.355202

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [35]:
new=df.groupby("State").aggregate({"Customer":"count"})
over_500= new[new["Customer"]>500]
print(over_500.sort_values)

<bound method DataFrame.sort_values of             Customer
State               
Arizona         1937
California      3552
Nevada           993
Oregon          2909
Washington       888>


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [40]:
df.groupby(["Education","Gender"]).aggregate({"Customer Lifetime Value":["max","min","mean"]})

Customer Lifetime Value                          
                                                max          min         mean
Education            Gender                                                  
Bachelor             F                  73225.95652  1904.000852  7874.269478
                     M                  67907.27050  1898.007675  7703.601675
College              F                  61850.18803  1898.683686  7748.823325
                     M                  61134.68307  1918.119700  8052.459288
Doctor               F                  44856.11397  2395.570000  7328.508916
                     M                  32677.34284  2267.604038  7415.333638
High School or Below F                  55277.44589  2144.921535  8675.220201
                     M                  83325.38119  1940.981221  8149.687783
Master               F                  51016.06704  2417.777032  8157.053154
                     M                  50568.25912  2272.307310  8168.832659

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [47]:
df["Effective To Date"]=pd.to_datetime(df["Effective To Date"])

C:\Users\claud\AppData\Local\Temp\ipykernel_1956\724080592.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Effective To Date"]=pd.to_datetime(df["Effective To Date"])


In [48]:
df["Month"] = df["Effective To Date"].dt.month_name()

In [55]:
policies_pivot=df.pivot_table(index="State",columns="Month",values="Number of Policies",aggfunc="sum")
month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]
policies_pivot = policies_pivot.reindex(columns=month_order, fill_value=0)

policies_pivot

Month,January,February,March,April,May,June,July,August,September,October,November,December
State,,,,,,,,,,,,
Arizona,3052,2864,0,0,0,0,0,0,0,0,0,0
California,5673,4929,0,0,0,0,0,0,0,0,0,0
Nevada,1493,1278,0,0,0,0,0,0,0,0,0,0
Oregon,4697,3969,0,0,0,0,0,0,0,0,0,0
Washington,1358,1225,0,0,0,0,0,0,0,0,0,0


Month,February,January
State,,
Arizona,2864,3052
California,4929,5673
Nevada,1278,1493
Oregon,3969,4697
Washington,1225,1358


In [50]:
df.groupby("State").aggregate({"Month","Number of Policies"})

AttributeError: 'SeriesGroupBy' object has no attribute 'Month'

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here